<div style="padding:28px;border-radius:18px;background:linear-gradient(135deg,#0f172a,#1e3a5f);color:white">
  <div style="font-size:13px;letter-spacing:2px;font-weight:700">TOPIC 766 · TUTORIAL OF AGENTIC SYSTEMS</div>
  <h1 style="margin:8px 0 6px 0;font-size:34px">Notebook 1 — Build the Primitives</h1>
  <div style="font-size:18px;opacity:.9">Agents → Tools → Skills · A single M&A agent learns how to act</div>
</div>

### Introduction

Notebook 0 built the world. Notebook 1 introduces the first entity that can act inside it.

Our common environment is a synthetic investment-banking universe containing 500 companies, financial data, strategic attributes, and unstructured evidence. Until now, however, the information has been passive. A spreadsheet can contain revenue, EBITDA, valuation multiples, financial-report excerpts, analyst notes, and rumors, but it does not decide what to inspect or why. The purpose of this notebook is to cross that boundary. We will build the smallest useful agentic system from three primitives: **an agent, deterministic tools, and reusable skills**.

The M&A problem gives these concepts concrete meaning. Imagine that a banker asks: “For Company C001, identify a plausible acquisition target and explain the recommendation.” A language model should not simply invent an answer from its internal knowledge. It should operate inside the synthetic world we created. It needs tools that can retrieve a company, screen candidates, compare financial metrics, and search textual evidence. It also needs a disciplined method for using those tools. That reusable method is what we will call a skill. Finally, the agent must decide which operation is needed next, interpret the observations returned by the tools, and stop when it has enough evidence to answer.

The objective is intentionally modest. We are not building a team of agents, a feedback loop, or a self-organizing architecture. Those capabilities come later. Here we want to understand one foundational question with complete transparency: **How does an individual agent acquire the ability to act?** By the end of the notebook, a single GPT-5.2 agent will receive an M&A mandate in ordinary language, choose among deterministic tools, follow an explicit M&A assessment skill, and produce a recommendation accompanied by an auditable trace of the operations it performed.

<div style="padding:20px;border-left:6px solid #2563eb;background:#eff6ff;border-radius:10px">
<b>What we need to understand and learn</b>
</div>

The first concept to understand is that **an agent is not the same thing as a language model**. A language model can transform an input into an output. An agentic system adds a control problem: given an objective and an environment, what should happen next? The model becomes useful as a decision-making component because it can interpret an ordinary-language objective, decide which available action is relevant, inspect the result of that action, and continue until it can produce an answer. Agency therefore appears through the interaction of reasoning, state, actions, and observations.

The second concept is the **tool**. A tool should do something definite. In this notebook, a tool may retrieve a company, filter the dataset, calculate a comparison, or search the document corpus. The language model does not calculate EBITDA multiples itself and it does not pretend to remember the contents of our CSV files. It asks a deterministic Python function to perform the operation. This division of labor is fundamental. Language models are flexible at interpretation and synthesis; deterministic code is preferable for exact filtering, arithmetic, lookup, validation, and other operations whose behavior should be predictable.

The third concept is the **skill**. A skill is not merely another function with a different name. It is a reusable method for solving a class of problems. Our M&A target-assessment skill will say, in effect: understand the buyer; inspect the buyer; screen candidates; compare structured evidence; inspect relevant text; reconcile contradictions; and produce a recommendation that separates facts from judgment. The individual tools remain small. The skill gives them procedural meaning. A useful analogy is that a calculator is a tool, while discounted-cash-flow valuation is a skill: the skill specifies how and why several operations should be combined.

The fourth concept is **tool choice**. We could hard-code a workflow saying “always call Tool A, then Tool B, then Tool C.” That would be a workflow, not yet a very interesting agent. Instead, the GPT-5.2 agent receives tool descriptions and the skill protocol, then chooses what to call based on the mandate and observations. We keep the action space deliberately small so the learner can inspect every decision.

The fifth concept is **auditability**. We do not need access to private chain-of-thought to understand what the system did. The notebook records an operational trace: which tool was called, with what arguments, and what result was returned. That is enough to study behavior at the level relevant to system design.

A sixth concept is **bounded autonomy**. The agent is autonomous only inside an action space we define. It cannot browse arbitrary files, change the dataset, call unknown functions, or inspect the teacher benchmark. This is a crucial lesson: useful agency is not equivalent to unlimited freedom. Good architecture gives an agent sufficient discretion to solve the problem while keeping the environment legible and governed.

Finally, we need to recognize the limitation of this architecture. One agent can perform the entire mandate, but it must juggle financial analysis, strategic reasoning, textual intelligence, and synthesis by itself. That limitation is deliberate. Once we experience it, Notebook 2 will have a natural motivation: **specialization and collaboration through constellations**.



| Concept | Characteristics | Example in M&A Tutorial | Key Distinction |
|:--------|:----------------|:------------------------|:------------------|
| **Agent** | Interprets objectives, decides next action, observes results, stops when objective met. The decision-making entity. | The GPT-5.2 model instructed by `run_ma_agent` to identify an M&A target. | Makes decisions and orchestrates actions. |
| **Tool** | Performs a definite, deterministic operation; has explicit inputs and predictable output; does not make judgments. | `get_company_snapshot()`: Retrieves structured data for a given `company_id`. | Performs specific, defined operations. |
| **Skill** | A reusable method or procedure for solving a class of problems; coordinates tools to achieve an objective. | `M&A Target Assessment Skill`: A protocol for sequencing operations (understand buyer, screen, compare, inspect evidence). | Provides a procedural guide for combining tools. |

### Code Unit 1 of 10 — Connect the notebook to the shared world

The first code unit reconnects the tutorial to the persistent environment created in Notebook 0. We mount Google Drive, point to the common `DATASET` directory, retrieve `OPENAI_API_KEY` from Colab Secrets, and centralize the requested model name as `gpt-5.2`. This is deliberately similar to the opening of Notebook 0 because consistency is part of the pedagogy. Students should immediately recognize what belongs to infrastructure and what belongs to the new agentic layer. The API key is never printed or written to Drive. We also create the OpenAI client but do not call the model yet. That separation is useful: merely initializing an agent environment should not spend tokens or create model behavior. By the end of this cell, we have access to two resources—the local synthetic M&A environment and a language model—but they are still disconnected. The remaining cells progressively construct the bridge between them.

In [1]:
%pip -q install -U openai

from google.colab import drive, userdata
from openai import OpenAI
from pathlib import Path
import pandas as pd
import numpy as np
import json, re

# Persist the tutorial dataset across Colab sessions.
drive.mount('/content/drive')

DATASET_DIR = Path(
    '/content/drive/MyDrive/Colab Notebooks/'
    'TOPIC_766 TUTORIAL OF AGENTIC SYSTEMS/DATASET'
)

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    raise ValueError('Add OPENAI_API_KEY in Colab → Secrets before continuing.')

MODEL = 'gpt-5.2'
client = OpenAI(api_key=OPENAI_API_KEY)

print('Dataset:', DATASET_DIR)
print('Model:', MODEL)
print('Environment initialized.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 17.3 MB/s eta 0:00:00
Mounted at /content/drive
Dataset: /content/drive/MyDrive/Colab Notebooks/TOPIC_766 TUTORIAL OF AGENTIC SYSTEMS/DATASET
Model: gpt-5.2
Environment initialized.


### Code Unit 2 of 10 — Load only the evidence the agent is allowed to see

An agent’s environment is defined partly by what information it can access. This cell loads the four student-facing files produced by Notebook 0: company identities, financials, strategic profiles, and documents. It intentionally does **not** load `teacher_mna_key.csv`. That file contains the synthetic benchmark we created for later evaluation, and allowing the agent to read it would collapse the exercise into a lookup. We then join the three structured tables into a convenient analytical view and perform a small contract check: 500 companies must exist, identifiers must be unique, and every company must have exactly three associated documents. This is the first example of a recurring design principle in the tutorial: before giving autonomy to a component, define and verify its environment. An agent cannot compensate for missing joins, duplicate identifiers, or corrupted data. The cell finishes by displaying one company and its three documents so that the learner sees the evidence directly before any tool abstraction is introduced.

In [2]:
REQUIRED_FILES = [
    'companies.csv',
    'financials.csv',
    'strategic_profiles.csv',
    'documents.csv',
]

missing = [f for f in REQUIRED_FILES if not (DATASET_DIR / f).exists()]
if missing:
    raise FileNotFoundError(f'Missing files: {missing}. Run Notebook 0 first.')

companies = pd.read_csv(DATASET_DIR / 'companies.csv')
financials = pd.read_csv(DATASET_DIR / 'financials.csv')
strategic_profiles = pd.read_csv(DATASET_DIR / 'strategic_profiles.csv')
documents = pd.read_csv(DATASET_DIR / 'documents.csv')

universe = (
    companies
    .merge(financials, on='company_id', validate='one_to_one')
    .merge(strategic_profiles, on='company_id', validate='one_to_one')
)

assert len(universe) == 500
assert universe['company_id'].is_unique
assert documents.groupby('company_id').size().eq(3).all()

example_id = 'C001'
display(universe[universe['company_id'] == example_id].T)
display(documents.loc[
    documents['company_id'] == example_id,
    ['document_type','source_reliability','text']
])

print('✓ Student-facing environment loaded and validated.')
print('✓ Teacher benchmark was not loaded.')

,0
company_id,C001
company_name,Delta Industries 001
country,Brazil
continent,South America
sector,Technology
subsector,Cybersecurity
founded_year,2000
ownership,Private
revenue_usd_m,1968.5
ebitda_usd_m,731.6


,document_type,source_reliability,text
0,financial_report_excerpt,High,Delta Industries 001 reported steady momentum ...
1,analyst_note,Moderate,Our channel checks suggest Delta Industries 00...
2,rumor,Low,There is speculative talk that Delta Industrie...


✓ Student-facing environment loaded and validated.
✓ Teacher benchmark was not loaded.


### Code Unit 3 of 10 — Build the first deterministic tool: retrieve a company snapshot

Our first tool answers a very simple question: “What do we know structurally about this company?” The function accepts a stable `company_id`, retrieves exactly one record from the joined universe, and returns a compact dictionary containing identity, geography, financial metrics, and strategic attributes. This may look trivial, but it establishes the tool contract that the rest of the notebook will follow. A tool has a name, a small set of explicit inputs, predictable behavior, and a structured output. It should fail clearly when the requested company does not exist rather than forcing the language model to infer what went wrong. Notice also what the tool does **not** do: it does not recommend whether the company should be bought. That judgment belongs elsewhere. Keeping tools narrow helps us reason about system behavior. If an M&A recommendation is poor, we can distinguish whether the problem arose from retrieval, screening, analysis, or synthesis rather than hiding all of those operations inside one opaque function.

In [3]:
SNAPSHOT_FIELDS = [
    'company_id','company_name','country','continent','sector','subsector','ownership',
    'revenue_usd_m','ebitda_usd_m','ebitda_margin_pct','revenue_growth_pct',
    'debt_usd_m','cash_usd_m','market_cap_usd_m','enterprise_value_usd_m',
    'ev_ebitda','net_debt_ebitda','business_model','strategic_strength',
    'strategic_weakness','geographic_posture','acquisition_appetite',
    'cross_border_openness','regulatory_sensitivity','integration_complexity'
]

def get_company_snapshot(company_id: str) -> dict:
    """Return one company's structured M&A profile."""
    match = universe.loc[universe['company_id'] == company_id, SNAPSHOT_FIELDS]
    if match.empty:
        return {'ok': False, 'error': f'Unknown company_id: {company_id}'}
    record = json.loads(match.to_json(orient='records'))[0]
    return {'ok': True, 'company': record}

snapshot = get_company_snapshot('C001')
print(json.dumps(snapshot, indent=2)[:2500])

{
  "ok": true,
  "company": {
    "company_id": "C001",
    "company_name": "Delta Industries 001",
    "country": "Brazil",
    "continent": "South America",
    "sector": "Technology",
    "subsector": "Cybersecurity",
    "ownership": "Private",
    "revenue_usd_m": 1968.5,
    "ebitda_usd_m": 731.6,
    "ebitda_margin_pct": 37.17,
    "revenue_growth_pct": 16.35,
    "debt_usd_m": 1500.5,
    "cash_usd_m": 104.6,
    "market_cap_usd_m": 10258.6,
    "enterprise_value_usd_m": 11654.6,
    "ev_ebitda": 15.93,
    "net_debt_ebitda": 1.91,
    "business_model": "distribution-led",
    "strategic_strength": "specialized talent base",
    "strategic_weakness": "elevated leverage",
    "geographic_posture": "Multi-region",
    "acquisition_appetite": "Moderate",
    "cross_border_openness": "Moderate",
    "regulatory_sensitivity": "Moderate",
    "integration_complexity": "Low"
  }
}


### Code Unit 4 of 10 — Build a screening tool that reduces the search space

M&A bankers rarely begin a target search by reading every company one by one. They reduce the universe using explicit criteria and then investigate the survivors. This cell creates a deterministic screening tool that can filter by sector and continent, exclude the buyer itself, impose minimum growth and maximum valuation or leverage thresholds, and rank the surviving firms with a transparent score. The ranking formula is intentionally simple. Its purpose is not to claim that M&A can be reduced to one number; its purpose is to give the agent a reproducible first-pass mechanism. This distinction is central to the pedagogical journey. A tool can narrow 500 companies to a manageable shortlist, but it does not understand the mandate in the broader sense. It cannot decide whether a high valuation is justified by strategic fit or whether a rumor should change the interpretation. Later the agent will decide when screening is useful and which criteria are appropriate. The tool itself remains deterministic.

In [4]:
def screen_candidates(
    exclude_company_id: str,
    sector: str,
    continent: str,
    min_growth_pct: float,
    max_ev_ebitda: float,
    max_net_debt_ebitda: float,
    limit: int,
) -> dict:
    """Filter and rank acquisition candidates using transparent criteria."""
    df = universe.copy()

    if exclude_company_id != 'NONE':
        df = df[df['company_id'] != exclude_company_id]
    if sector != 'ANY':
        df = df[df['sector'] == sector]
    if continent != 'ANY':
        df = df[df['continent'] == continent]

    df = df[
        (df['revenue_growth_pct'] >= min_growth_pct) &
        (df['ev_ebitda'] <= max_ev_ebitda) &
        (df['net_debt_ebitda'] <= max_net_debt_ebitda)
    ].copy()

    if df.empty:
        return {'ok': True, 'count': 0, 'candidates': []}

    df['screen_score'] = (
        0.50 * df['revenue_growth_pct']
        - 0.30 * df['ev_ebitda']
        - 0.20 * df['net_debt_ebitda']
    )

    cols = [
        'company_id','company_name','country','continent','sector','subsector',
        'revenue_growth_pct','ebitda_margin_pct','ev_ebitda',
        'net_debt_ebitda','screen_score'
    ]
    result = df.sort_values('screen_score', ascending=False).head(int(limit))[cols]
    return {
        'ok': True,
        'count': len(result),
        'candidates': json.loads(result.to_json(orient='records')),
        'ranking_note': '0.50*growth - 0.30*EV/EBITDA - 0.20*net debt/EBITDA'
    }

demo_screen = screen_candidates('C001','Technology','ANY',5.0,24.0,4.0,5)
display(pd.DataFrame(demo_screen['candidates']))

,company_id,company_name,country,continent,sector,subsector,revenue_growth_pct,ebitda_margin_pct,ev_ebitda,net_debt_ebitda,screen_score
0,C141,Horizon Group 141,Egypt,Africa,Technology,Cybersecurity,29.06,30.22,17.15,-0.10,9.405
1,C021,Terra Group 021,Mexico,North America,Technology,IT Services,29.32,33.06,18.21,0.03,9.191
2,C441,Horizon Group 441,New Zealand,Oceania,Technology,Cybersecurity,27.55,32.61,14.94,0.87,9.119
3,C341,Horizon Global 341,United States,North America,Technology,Semiconductors,28.23,21.78,17.80,0.49,8.677
4,C491,Horizon Networks 491,Australia,Oceania,Technology,IT Services,27.47,29.48,18.64,-0.16,8.175


### Code Unit 5 of 10 — Build a tool for searching unstructured evidence

Structured screening tells us what the numbers say. M&A work also requires reading what people say. This cell creates a deliberately simple document-search tool over the 1,500 synthetic text records. We avoid embeddings, vector databases, and retrieval frameworks at this stage because the tutorial is about primitives. A transparent lexical score is easier to understand: the query is tokenized, the documents are tokenized, and each overlapping term contributes to the score. The search can be restricted to one company or one document type, or it can span the whole corpus. The returned result includes document type and source reliability because an analyst should not treat a low-reliability rumor as equivalent to a financial-report excerpt. Later notebooks can replace this simple retrieval method with more sophisticated search without changing the conceptual role of the tool. The important lesson is that the agent does not “remember” our synthetic documents. It must perform an explicit retrieval action and then reason over the evidence returned.

In [5]:
def _tokens(text: str) -> list[str]:
    return re.findall(r'[a-z0-9]+', str(text).lower())

def search_documents(query: str, company_id: str, document_type: str, limit: int) -> dict:
    """Search the synthetic document corpus with a transparent lexical score."""
    df = documents.copy()

    if company_id != 'ANY':
        df = df[df['company_id'] == company_id]
    if document_type != 'ANY':
        df = df[df['document_type'] == document_type]

    query_tokens = [t for t in _tokens(query) if len(t) > 2]
    if not query_tokens:
        return {'ok': False, 'error': 'Query contains no usable terms.'}

    def score(text):
        doc_tokens = _tokens(text)
        return sum(doc_tokens.count(token) for token in query_tokens)

    df['search_score'] = df['text'].map(score)
    df = df[df['search_score'] > 0].sort_values('search_score', ascending=False)

    result = df.head(int(limit))[[
        'document_id','company_id','document_type','source_reliability',
        'search_score','text'
    ]]
    return {
        'ok': True,
        'query': query,
        'count': len(result),
        'documents': json.loads(result.to_json(orient='records'))
    }

demo_docs = search_documents(
    'strategic interest acquisition consolidation growth', 'C001', 'ANY', 5
)
display(pd.DataFrame(demo_docs['documents']))

,document_id,company_id,document_type,source_reliability,search_score,text
0,D0003,C001,rumor,Low,2,There is speculative talk that Delta Industrie...


### Code Unit 6 of 10 — Build a comparison tool for a shortlist

A shortlist is useful only if we can compare candidates consistently. This cell creates a fourth deterministic tool that accepts a list of company IDs and returns a common comparison table. The function validates every identifier, selects a compact set of financial and strategic variables, and calculates three normalized teaching indicators: growth attractiveness, valuation attractiveness, and balance-sheet attractiveness. It then combines them into a simple structured score. Again, the score is not presented as an investment-banking truth. It is a transparent analytical device that allows the learner to distinguish computation from judgment. The tool can say that Candidate A has stronger growth, a cheaper multiple, or lower leverage than Candidate B. It cannot decide whether Candidate B’s geographic position or proprietary technology outweighs those disadvantages. That synthesis belongs to the agent operating under a skill. By this point, we have four small capabilities that are independently testable. None is an agent. Together, however, they form the action vocabulary from which an agent can begin to operate.

In [6]:
def compare_candidates(company_ids: list[str]) -> dict:
    """Compare a shortlist on common financial and strategic dimensions."""
    requested = list(dict.fromkeys(company_ids))
    known = set(universe['company_id'])
    unknown = [cid for cid in requested if cid not in known]

    if unknown:
        return {'ok': False, 'error': f'Unknown company IDs: {unknown}'}
    if len(requested) < 2:
        return {'ok': False, 'error': 'Provide at least two company IDs.'}

    cols = [
        'company_id','company_name','country','sector','subsector',
        'revenue_growth_pct','ebitda_margin_pct','ev_ebitda','net_debt_ebitda',
        'strategic_strength','strategic_weakness','regulatory_sensitivity',
        'integration_complexity'
    ]
    df = universe[universe['company_id'].isin(requested)][cols].copy()

    df['growth_attractiveness'] = np.clip((df['revenue_growth_pct'] + 10) / 40, 0, 1)
    df['valuation_attractiveness'] = np.clip((26 - df['ev_ebitda']) / 21, 0, 1)
    df['balance_sheet_attractiveness'] = np.clip((6 - df['net_debt_ebitda']) / 6, 0, 1)
    df['structured_score'] = (
        0.40 * df['growth_attractiveness'] +
        0.35 * df['valuation_attractiveness'] +
        0.25 * df['balance_sheet_attractiveness']
    )
    df = df.sort_values('structured_score', ascending=False)

    return {
        'ok': True,
        'comparison': json.loads(df.to_json(orient='records')),
        'warning': 'Structured score is a teaching aid, not a complete M&A recommendation.'
    }

screen_ids = [x['company_id'] for x in demo_screen['candidates'][:3]]
comparison_demo = compare_candidates(screen_ids)
display(pd.DataFrame(comparison_demo['comparison']))

,company_id,company_name,country,sector,subsector,revenue_growth_pct,ebitda_margin_pct,ev_ebitda,net_debt_ebitda,strategic_strength,strategic_weakness,regulatory_sensitivity,integration_complexity,growth_attractiveness,valuation_attractiveness,balance_sheet_attractiveness,structured_score
0,C141,Horizon Group 141,Egypt,Technology,Cybersecurity,29.06,30.22,17.15,-0.10,specialized talent base,margin pressure,Low,Moderate,0.97650,0.421429,1.000,0.788100
1,C441,Horizon Group 441,New Zealand,Technology,Cybersecurity,27.55,32.61,14.94,0.87,strong distribution network,margin pressure,Moderate,High,0.93875,0.526667,0.855,0.773583
2,C021,Terra Group 021,Mexico,Technology,IT Services,29.32,33.06,18.21,0.03,trusted brand,slow international expansion,High,High,0.98300,0.370952,0.995,0.771783


### Code Unit 7 of 10 — Encode a reusable skill rather than another tool

We now introduce the concept that most clearly distinguishes this notebook from a collection of utility functions. The `M&A Target Assessment Skill` is a reusable protocol describing how an analyst should approach a target-selection mandate. It does not calculate anything itself. Instead, it tells the agent how to sequence attention: understand the buyer, screen broadly, compare a shortlist, inspect unstructured evidence, reconcile contradictions, and state a recommendation with uncertainty. The cell also provides a small deterministic demonstration that follows the first three steps of the skill so the learner can see the difference between a **procedure** and an **operation**. This distinction will matter enormously later. Tools can be shared by many skills; a skill can coordinate several tools; and an agent can choose which skill is appropriate to an objective. We deliberately keep the skill as readable text and a visible protocol rather than hiding it inside an agent framework. The student should be able to inspect and modify the method before giving the model any autonomy.

In [7]:
MA_TARGET_ASSESSMENT_SKILL = """
M&A TARGET ASSESSMENT SKILL — VERSION 1.0

1. UNDERSTAND THE BUYER
   Retrieve the buyer snapshot and identify sector, geography, financial capacity,
   strategic strengths, weaknesses, and cross-border posture.

2. CREATE A SHORTLIST
   Screen the universe using explicit criteria appropriate to the mandate.
   Never treat the screening score as the final answer.

3. COMPARE STRUCTURED EVIDENCE
   Compare at least two plausible targets on growth, valuation, leverage,
   strategic strengths, regulatory sensitivity, and integration complexity.

4. INSPECT UNSTRUCTURED EVIDENCE
   Search financial-report excerpts, analyst notes, and rumors for the leading
   candidates. Treat source reliability explicitly.

5. RECONCILE CONTRADICTIONS
   Explain disagreements between numerical and textual evidence. Never turn an
   unverified rumor into a fact.

6. RECOMMEND
   Name a preferred target, identify an alternative, explain the evidence,
   state risks, and distinguish facts from judgment.

7. STOP
   Stop when the recommendation is sufficiently supported. Do not call tools
   merely to appear thorough.
"""

print(MA_TARGET_ASSESSMENT_SKILL)

# Deterministic preview: a skill coordinates several tools.
buyer_demo = get_company_snapshot('C001')
buyer_sector = buyer_demo['company']['sector']
shortlist_demo = screen_candidates('C001', buyer_sector, 'ANY', 5.0, 24.0, 4.0, 3)
ids_demo = [x['company_id'] for x in shortlist_demo['candidates']]

if len(ids_demo) >= 2:
    display(pd.DataFrame(compare_candidates(ids_demo)['comparison']))
else:
    print('Relax demo thresholds if fewer than two targets are returned.')


M&A TARGET ASSESSMENT SKILL — VERSION 1.0

1. UNDERSTAND THE BUYER
   Retrieve the buyer snapshot and identify sector, geography, financial capacity,
   strategic strengths, weaknesses, and cross-border posture.

2. CREATE A SHORTLIST
   Screen the universe using explicit criteria appropriate to the mandate.
   Never treat the screening score as the final answer.

3. COMPARE STRUCTURED EVIDENCE
   Compare at least two plausible targets on growth, valuation, leverage,
   strategic strengths, regulatory sensitivity, and integration complexity.

4. INSPECT UNSTRUCTURED EVIDENCE
   Search financial-report excerpts, analyst notes, and rumors for the leading
   candidates. Treat source reliability explicitly.

5. RECONCILE CONTRADICTIONS
   Explain disagreements between numerical and textual evidence. Never turn an
   unverified rumor into a fact.

6. RECOMMEND
   Name a preferred target, identify an alternative, explain the evidence,
   state risks, and distinguish facts from judgment.

7.

,company_id,company_name,country,sector,subsector,revenue_growth_pct,ebitda_margin_pct,ev_ebitda,net_debt_ebitda,strategic_strength,strategic_weakness,regulatory_sensitivity,integration_complexity,growth_attractiveness,valuation_attractiveness,balance_sheet_attractiveness,structured_score
0,C141,Horizon Group 141,Egypt,Technology,Cybersecurity,29.06,30.22,17.15,-0.10,specialized talent base,margin pressure,Low,Moderate,0.97650,0.421429,1.000,0.788100
1,C441,Horizon Group 441,New Zealand,Technology,Cybersecurity,27.55,32.61,14.94,0.87,strong distribution network,margin pressure,Moderate,High,0.93875,0.526667,0.855,0.773583
2,C021,Terra Group 021,Mexico,Technology,IT Services,29.32,33.06,18.21,0.03,trusted brand,slow international expansion,High,High,0.98300,0.370952,0.995,0.771783


### Code Unit 8 of 10 — Describe the tools to the language model and create an execution registry

Python functions do not automatically become actions that a language model can select. We need an interface between the model’s decision and the deterministic environment. This cell creates that interface in two parts. First, each tool receives a JSON schema describing its name, purpose, arguments, and allowed values. This is the action vocabulary visible to GPT-5.2. Second, a local registry maps each tool name to the actual Python function that executes it. Keeping these two layers separate is useful for governance: the model can request only operations that have been explicitly described, and the runtime executes only names that have been explicitly registered. We use strict schemas and intentionally small argument sets to reduce ambiguity. Notice that the M&A skill is **not** exposed as another callable function. Instead, it will become part of the agent’s instructions. This preserves the conceptual separation: the skill tells the agent how to work; the tools are the concrete actions the agent may invoke.

In [8]:
OPENAI_TOOLS = [
    {
        'type':'function',
        'name':'get_company_snapshot',
        'description':'Retrieve structured financial and strategic data for one synthetic company.',
        'parameters':{
            'type':'object',
            'properties':{'company_id':{'type':'string','description':'Stable company ID such as C001.'}},
            'required':['company_id'],
            'additionalProperties':False,
        },
        'strict':True,
    },
    {
        'type':'function',
        'name':'screen_candidates',
        'description':'Filter and rank possible M&A targets using explicit structured criteria.',
        'parameters':{
            'type':'object',
            'properties':{
                'exclude_company_id':{'type':'string'},
                'sector':{'type':'string','description':'Exact sector or ANY.'},
                'continent':{'type':'string','description':'Exact continent or ANY.'},
                'min_growth_pct':{'type':'number'},
                'max_ev_ebitda':{'type':'number'},
                'max_net_debt_ebitda':{'type':'number'},
                'limit':{'type':'integer'},
            },
            'required':['exclude_company_id','sector','continent','min_growth_pct','max_ev_ebitda','max_net_debt_ebitda','limit'],
            'additionalProperties':False,
        },
        'strict':True,
    },
    {
        'type':'function',
        'name':'search_documents',
        'description':'Search synthetic financial-report excerpts, analyst notes, and M&A rumors.',
        'parameters':{
            'type':'object',
            'properties':{
                'query':{'type':'string'},
                'company_id':{'type':'string','description':'Company ID or ANY.'},
                'document_type':{'type':'string','enum':['ANY','financial_report_excerpt','analyst_note','rumor']},
                'limit':{'type':'integer'},
            },
            'required':['query','company_id','document_type','limit'],
            'additionalProperties':False,
        },
        'strict':True,
    },
    {
        'type':'function',
        'name':'compare_candidates',
        'description':'Compare two or more target companies using common structured dimensions.',
        'parameters':{
            'type':'object',
            'properties':{
                'company_ids':{
                    'type':'array','items':{'type':'string'}
                }
            },
            'required':['company_ids'],
            'additionalProperties':False,
        },
        'strict':True,
    },
]

TOOL_REGISTRY = {
    'get_company_snapshot': get_company_snapshot,
    'screen_candidates': screen_candidates,
    'search_documents': search_documents,
    'compare_candidates': compare_candidates,
}

print('Tools available to the agent:')
for tool in OPENAI_TOOLS:
    print(' -', tool['name'])

Tools available to the agent:
 - get_company_snapshot
 - screen_candidates
 - search_documents
 - compare_candidates


### Code Unit 9 of 10 — Build the single agent and its observe–decide–act cycle

This cell finally creates the agentic control loop. The agent receives three things: a mandate written in ordinary language, the M&A assessment skill, and descriptions of the four tools. GPT-5.2 can either answer or request a tool call. When it requests a tool, Python validates the tool name, executes the corresponding deterministic function, records the operation in an audit log, and returns the observation to the model. The model then decides what to do next. This repeats until the model produces a final answer or a safety limit on rounds is reached. The loop is intentionally visible and short; we are not using a high-level agent framework because that would conceal the primitive we are trying to understand. Also note what the audit trace contains: operational actions and outputs, not private chain-of-thought. For system engineering, that is the relevant evidence. We can see whether the agent retrieved the buyer, screened candidates, searched documents, and compared targets without requiring access to hidden internal reasoning.

In [9]:
AGENT_INSTRUCTIONS = f"""
You are a pedagogical M&A investment-banking agent operating ONLY on the
synthetic TOPIC 766 dataset.

Follow this reusable skill:

{MA_TARGET_ASSESSMENT_SKILL}

Rules:
- Never invent company facts.
- Use tools for dataset facts, filtering, comparisons, and document retrieval.
- Never access or mention any teacher benchmark.
- Distinguish low-reliability rumors from higher-reliability evidence.
- Screening and structured scores are analytical aids, not ground truth.
- Keep the final answer concise but evidence-based.
- Cite company IDs whenever naming a synthetic company.
"""

def run_ma_agent(mission: str, max_rounds: int = 8) -> dict:
    audit_log = []

    response = client.responses.create(
        model=MODEL,
        instructions=AGENT_INSTRUCTIONS,
        input=mission,
        tools=OPENAI_TOOLS,
        parallel_tool_calls=False,
    )

    for round_number in range(1, max_rounds + 1):
        calls = [
            item for item in response.output
            if getattr(item, 'type', None) == 'function_call'
        ]

        if not calls:
            return {
                'status':'completed',
                'answer':response.output_text,
                'audit_log':audit_log,
                'response_id':response.id,
            }

        tool_outputs = []
        for call in calls:
            name = call.name
            args = json.loads(call.arguments)
            result = (
                TOOL_REGISTRY[name](**args)
                if name in TOOL_REGISTRY
                else {'ok':False,'error':f'Tool not registered: {name}'}
            )

            audit_log.append({
                'round':round_number,
                'tool':name,
                'arguments':args,
                'result':result,
            })

            tool_outputs.append({
                'type':'function_call_output',
                'call_id':call.call_id,
                'output':json.dumps(result, ensure_ascii=False),
            })

        # Repeat governing instructions on every continuation call.
        response = client.responses.create(
            model=MODEL,
            instructions=AGENT_INSTRUCTIONS,
            previous_response_id=response.id,
            input=tool_outputs,
            tools=OPENAI_TOOLS,
            parallel_tool_calls=False,
        )

    return {
        'status':'max_rounds_reached',
        'answer':response.output_text,
        'audit_log':audit_log,
        'response_id':response.id,
    }

print('Single-agent runtime is ready.')

Single-agent runtime is ready.


### Code Unit 10 of 10 — Run an M&A mandate and inspect the agent’s operational trace

The final code unit turns the architecture into an experiment. We give the agent a natural-language mandate involving buyer `C001`: identify a plausible acquisition target, consider several candidates, inspect both structured and unstructured evidence, and present an alternative. We do not tell the agent exactly which tools to call or in what order. That is the point of the exercise. The model must interpret the objective under the M&A skill and choose actions from the limited vocabulary we built. After the run, the notebook prints the final recommendation and a compact audit table showing the tool sequence. The learner should compare these two artifacts. The answer shows what the agent concluded; the trace shows what the system actually did. If the recommendation is weak, we now have specific questions to ask: Did it screen too narrowly? Did it fail to inspect a rumor? Did it compare too few candidates? This is the first time in the tutorial that behavior emerges from a model choosing among actions rather than from a completely predetermined workflow.

In [10]:
MISSION = """
You are advising synthetic company C001 on a possible acquisition.

Identify one plausible acquisition target from the synthetic dataset.
Prefer strategic coherence with the buyer, but do not rely on financial
screening alone. Consider at least two serious candidates, inspect relevant
unstructured evidence for the leading candidates, and identify one alternative
to the preferred target.

Your final recommendation must contain:
1. preferred target and company ID,
2. concise financial and strategic rationale,
3. material textual evidence and its reliability,
4. main risk or contradiction,
5. one alternative target,
6. a short statement of what remains uncertain.
"""

RESULT = run_ma_agent(MISSION)

print('=' * 90)
print('AGENT STATUS:', RESULT['status'])
print('=' * 90)
print(RESULT['answer'])

audit_rows = []
for event in RESULT['audit_log']:
    r = event['result']
    if isinstance(r, dict) and 'count' in r:
        summary = f"count={r['count']}"
    elif isinstance(r, dict) and 'ok' in r:
        summary = f"ok={r['ok']}"
    else:
        summary = 'result returned'

    audit_rows.append({
        'round': event['round'],
        'tool': event['tool'],
        'arguments': json.dumps(event['arguments'], ensure_ascii=False),
        'result_summary': summary,
    })

print('\nOPERATIONAL AUDIT TRACE')
display(pd.DataFrame(audit_rows))

print('\nPEDAGOGICAL CHECK')
print('The agent had tools + a skill, but only one reasoning role.')
print('That limitation motivates NB02: specialized agents collaborating as a constellation.')

AGENT STATUS: completed
## 1) Preferred target (company ID)
**Horizon Group 141 (C141)** — Egypt (Technology / Cybersecurity)

## 2) Concise financial + strategic rationale (why C141 fits C001)
- **Strategic coherence:** Both are **Cybersecurity** players; C001’s stated strength is a **specialized talent base**, which matches C141’s **specialized talent base** and suggests potential to scale delivery capacity and product/service depth rather than forcing an adjacency.
- **Cross-border logic:** C001 is **multi-region** with **moderate cross-border openness**; Egypt provides a credible **EMEA/MENA foothold** while staying in-sector (lower “strategy drift”).
- **Financial profile (structured):** C141 shows **high growth (29.06%)**, solid **EBITDA margin (30.22%)**, and **very low leverage (net debt/EBITDA -0.1)**, which matters because C001’s own weakness is **elevated leverage (net debt/EBITDA 1.91)**—a target with a cleaner balance sheet reduces financing stress.
- **Valuation sanity:**

,round,tool,arguments,result_summary
0,1,get_company_snapshot,"{""company_id"": ""C001""}",ok=True
1,2,screen_candidates,"{""exclude_company_id"": ""C001"", ""sector"": ""Tech...",count=8
2,3,compare_candidates,"{""company_ids"": [""C141"", ""C441"", ""C351""]}",ok=True
3,4,search_documents,"{""query"": ""acquisition target strategic fit De...",count=12



PEDAGOGICAL CHECK
The agent had tools + a skill, but only one reasoning role.
That limitation motivates NB02: specialized agents collaborating as a constellation.


### The main take-out: The Building Blocks of Agency

To reiterate the fundamental concepts demonstrated in this notebook, we've established a clear distinction between an **Agent**, **Tools**, and **Skills**, which collectively enable an intelligent system to act purposefully:

*   An **Agent** is the decision-making entity. It interprets objectives, decides which actions to take, observes the results of those actions, and continues until its objective is met. In our M&A example, this is the GPT-5.2 model, guided by `run_ma_agent`, which orchestrates the entire process of identifying a target.

*   **Tools** are the deterministic operations the agent can invoke. Each tool performs a specific, well-defined function with explicit inputs and predictable outputs. They do not make judgments or interpret objectives; they simply execute. Examples include `get_company_snapshot()` for retrieving structured data or `screen_candidates()` for filtering the universe. Tools provide the concrete actions available to the agent.

*   A **Skill** represents a reusable method or procedure for solving a class of problems. It coordinates the use of various tools to achieve an objective by providing a structured protocol or set of guidelines. The `M&A Target Assessment Skill`, for instance, dictates the sequence of analytical steps (e.g., understand buyer, screen, compare, inspect evidence). While tools define *what* the agent *can do*, a skill defines *how* the agent *should do it* for a particular type of problem.

This division of labor is crucial: the agent focuses on high-level reasoning and decision-making, tools handle precise, auditable operations, and skills provide the procedural intelligence that guides the agent's actions through complex tasks.

##Conclusion
Notebook 1 has made the first genuine transition in our pedagogical ladder. Notebook 0 created an environment; Notebook 1 created a system that can act inside that environment. The change did not require a large framework or a complex autonomous architecture. It required only three primitives and a visible control loop.

The first primitive was the **tool**. We built small deterministic functions for retrieving a company, screening the universe, searching unstructured documents, and comparing candidates. Their importance lies less in their sophistication than in their clarity. Each operation has explicit inputs, a predictable output, and a narrow responsibility. The language model is therefore not asked to calculate everything, remember the dataset, or fabricate missing facts. It delegates exact operations to code.

The second primitive was the **skill**. The M&A Target Assessment Skill encoded a reusable method rather than an isolated operation. It instructed the system to understand the buyer, create a shortlist, compare structured evidence, inspect unstructured evidence, reconcile contradictions, recommend, and stop. This gave us an important conceptual separation: tools define what the system **can do**; a skill defines a disciplined way of combining those capabilities to solve a recurring class of problems.

The third primitive was the **agent**. GPT-5.2 received an objective, the skill, and descriptions of the tools. It then participated in an observe–decide–act cycle. When it needed information, it requested a deterministic operation. Python executed that operation and returned an observation. The model then decided whether another action was necessary or whether it had enough evidence to answer. Agency therefore appeared not as a mystical property but as a concrete mechanism: an objective, a decision-maker, an action space, observations, and a stopping condition.

The audit trace is equally important. We did not need to inspect private reasoning to understand the system operationally. We could see the sequence of tool calls, the arguments used, and the observations returned. This gives us a basis for debugging and governance. If an answer is unsatisfactory, we can investigate observable behavior. Did the agent retrieve the correct buyer? Did it screen with sensible thresholds? Did it compare multiple candidates? Did it inspect textual evidence? That is already a significant improvement over treating the language model as a black-box answer generator.

The notebook also introduced **bounded autonomy**. The agent could choose among actions, but only inside the environment we deliberately exposed. It could not alter the dataset, inspect the teacher benchmark, or call arbitrary functions. This matters because agentic design is not simply about giving models more freedom. It is about choosing the right boundary between discretion and control.

But the notebook should leave us dissatisfied in a productive way. The single agent is responsible for everything. It must interpret the mandate, think about financial attractiveness, strategic fit, source reliability, risk, and synthesis. Even though it has several tools, it has only one reasoning role. As the problem becomes richer, this creates cognitive and organizational pressure. We can make the prompt longer, but that does not create genuine specialization. We can add more tools, but a larger toolbox does not itself create a team.

That limitation gives us the natural bridge to Notebook 2. Instead of asking one agent to behave simultaneously like a valuation banker, a strategic analyst, an intelligence researcher, and a senior deal lead, we will separate those responsibilities. Several agents will receive distinct roles and potentially different subsets of tools. They will produce partial analyses and then collaborate to reach a combined recommendation.

The next conceptual transition is therefore:

**One Agent + Tools + Skill**

↓

**Specialized Agents + Roles + Collaboration**

↓

**Constellation**

The central question for Notebook 2 will be:

> **How can multiple specialized agents work together as one system—and what do we gain by distributing cognition rather than simply making one agent more complicated?**